In [0]:
import importlib.util

package_name = "lightgbm"

spec = importlib.util.find_spec(package_name)

if spec is None:
    print("lightgbm not installed. Installing...")
    %pip install lightgbm
    dbutils.library.restartPython()
else:
    print("lightgbm already installed.")

In [0]:
# Databricks notebook source
# ==========================================
# 03_ml_train.py
# NYC Taxi trip duration prediction
# LightGBM baseline with feature engineering + MLflow
# ==========================================

import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np

from math import sqrt, radians, sin, cos, asin
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from mlflow.models import infer_signature

from lightgbm import LGBMRegressor

import pandas as pd

FEATURE_SCHEMA = {
    "vendor_name": "string",
    "passenger_count": "Int",
    "trip_distance": "float",
    "rate_code": "string",
    "payment_type": "string",
    "fare_amount": "float",
    "tip_amount": "float",
    "total_amount": "float",
    "surcharge": "float",
    "mta_tax": "float",
    "tolls_amount": "float",
    "start_lon": "float",
    "start_lat": "float",
    "end_lon": "float",
    "end_lat": "float",
    "pickup_hour": "Int",
    "pickup_day_of_week": "Int",
    "pickup_month": "Int",
    "pickup_is_weekend": "Int",
    "avg_speed_mph": "float",
    "label": "float",
    "is_rush_hour": "Int",
    "is_night": "Int",
    "haversine_distance": "float",
    "manhattan_distance": "float",
    "fare_per_mile": "float",
    "tip_ratio": "float",
    "tolls_ratio": "float",
    "predicted_trip_duration": "float"
}

def apply_feature_schema(df: pd.DataFrame, schema_map: dict = FEATURE_SCHEMA) -> pd.DataFrame:
    df = df.copy()

    for col, dtype in schema_map.items():
        if col not in df.columns:
            continue

        if dtype == "string":
            df[col] = df[col].apply(lambda x: "unknown" if pd.isna(x) else str(x)).astype("string")

        elif dtype == "Int":
            df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int")

        elif dtype == "float":
            df[col] = pd.to_numeric(df[col], errors="coerce").astype("float")

    return df

# ------------------------------------------
# 1. Config
# ------------------------------------------
CATALOG = "nyctaxi_dev"
SCHEMA = "gold"
TABLE = "gld_taxi_training_features"

FULL_TABLE_NAME = f"{CATALOG}.{SCHEMA}.{TABLE}"
EXPERIMENT_PATH = "/Shared/nyctaxi_trip_duration_experiment"
REGISTERED_MODEL_NAME = "nyctaxi_dev.ml.nyctaxi_trip_duration_lgbm_model"

RANDOM_STATE = 42
TEST_SIZE = 0.2

# ------------------------------------------
# 2. Load data
# ------------------------------------------
df = spark.table(FULL_TABLE_NAME).limit(5000000).toPandas()
df = apply_feature_schema(df)

print(f"Loaded rows: {len(df):,}")
print("Columns:", df.columns.tolist())
print("Dtypes:")
print(df.dtypes)

# ------------------------------------------
# 3. Validate columns
# ------------------------------------------
required_columns = [
    "vendor_name",
    "passenger_count",
    "trip_distance",
    "rate_code",
    "payment_type",
    "fare_amount",
    "tip_amount",
    "total_amount",
    "surcharge",
    "mta_tax",
    "tolls_amount",
    "start_lon",
    "start_lat",
    "end_lon",
    "end_lat",
    "pickup_hour",
    "pickup_day_of_week",
    "pickup_month",
    "pickup_is_weekend",
    "label"
]

missing_cols = [c for c in required_columns if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns in {FULL_TABLE_NAME}: {missing_cols}")

# ------------------------------------------
# 4. Feature engineering
# ------------------------------------------

# 4.1 rush hour
df["is_rush_hour"] = df["pickup_hour"].isin([7, 8, 9, 16, 17, 18, 19]).astype(int)

# 4.2 night
df["is_night"] = ((df["pickup_hour"] < 6) | (df["pickup_hour"] >= 22)).astype(int)

# 4.3 haversine distance
def haversine_np(lon1, lat1, lon2, lat2):
    """
    Calculate great circle distance in kilometers.
    """
    lon1, lat1, lon2, lat2 = map(
        np.radians,
        [lon1, lat1, lon2, lat2]
    )

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    c = 2 * np.arcsin(np.sqrt(a))
    km = 6371 * c
    return km

df["haversine_distance"] = haversine_np(
    df["start_lon"],
    df["start_lat"],
    df["end_lon"],
    df["end_lat"]
)

# 4.4 manhattan-like distance
df["manhattan_distance"] = (
    haversine_np(df["start_lon"], df["start_lat"], df["end_lon"], df["start_lat"]) +
    haversine_np(df["start_lon"], df["start_lat"], df["start_lon"], df["end_lat"])
)

# 4.5 fare per mile
df["fare_per_mile"] = np.where(
    df["trip_distance"] > 0,
    df["fare_amount"] / df["trip_distance"],
    np.nan
)

# 4.6 tip ratio
df["tip_ratio"] = np.where(
    df["fare_amount"] > 0,
    df["tip_amount"] / df["fare_amount"],
    np.nan
)

# 4.7 tolls ratio
df["tolls_ratio"] = np.where(
    df["total_amount"] > 0,
    df["tolls_amount"] / df["total_amount"],
    np.nan
)

# ------------------------------------------
# 5. Data filtering
# ------------------------------------------
df = df[
    (df["label"].notna()) &
    (df["label"] > 0) &
    (df["label"] <= 180) &
    (df["trip_distance"].notna()) &
    (df["trip_distance"] > 0)
].copy()

# optional geographic sanity filters
df = df[
    df["start_lon"].between(-80, -70) &
    df["end_lon"].between(-80, -70) &
    df["start_lat"].between(35, 45) &
    df["end_lat"].between(35, 45)
].copy()

print(f"Rows after filtering: {len(df):,}")

# ------------------------------------------
# 6. Feature selection
# ------------------------------------------
target = "label"

feature_cols = [
    "vendor_name",
    "passenger_count",
    "trip_distance",
    "rate_code",
    "payment_type",
    "fare_amount",
    "tip_amount",
    "total_amount",
    "surcharge",
    "mta_tax",
    "tolls_amount",
    "start_lon",
    "start_lat",
    "end_lon",
    "end_lat",
    "pickup_hour",
    "pickup_day_of_week",
    "pickup_month",
    "pickup_is_weekend",
    "is_rush_hour",
    "is_night",
    "haversine_distance",
    "manhattan_distance",
    "fare_per_mile",
    "tip_ratio",
    "tolls_ratio"
]

categorical_cols = [
    "vendor_name",
    "rate_code",
    "payment_type"
]

numeric_cols = [
    "passenger_count",
    "trip_distance",
    "fare_amount",
    "tip_amount",
    "total_amount",
    "surcharge",
    "mta_tax",
    "tolls_amount",
    "start_lon",
    "start_lat",
    "end_lon",
    "end_lat",
    "pickup_hour",
    "pickup_day_of_week",
    "pickup_month",
    "pickup_is_weekend",
    "is_rush_hour",
    "is_night",
    "haversine_distance",
    "manhattan_distance",
    "fare_per_mile",
    "tip_ratio",
    "tolls_ratio"
]

df = df[feature_cols + [target]].copy()

# ------------------------------------------
# 7. Train / test split
# ------------------------------------------
X = df[feature_cols]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE
)

print(f"Train rows: {len(X_train):,}")
print(f"Test rows : {len(X_test):,}")

# ------------------------------------------
# 8. Preprocessing
# ------------------------------------------
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols)
    ]
)

# ------------------------------------------
# 9. LightGBM baseline
# ------------------------------------------
model = LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=-1,
    num_leaves=64,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_STATE
)

pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", model)
])

# ------------------------------------------
# 10. MLflow
# ------------------------------------------
mlflow.set_experiment(EXPERIMENT_PATH)

with mlflow.start_run() as run:
    mlflow.set_tag("project", "databricks-end-to-end-ml-nyctaxi")
    mlflow.set_tag("model_type", "LightGBM")
    mlflow.set_tag("dataset", FULL_TABLE_NAME)
    mlflow.set_tag("target", target)

    mlflow.log_param("test_size", TEST_SIZE)
    mlflow.log_param("random_state", RANDOM_STATE)
    mlflow.log_param("n_estimators", 500)
    mlflow.log_param("learning_rate", 0.05)
    mlflow.log_param("max_depth", -1)
    mlflow.log_param("num_leaves", 64)
    mlflow.log_param("subsample", 0.8)
    mlflow.log_param("colsample_bytree", 0.8)
    mlflow.log_param("feature_count", len(feature_cols))
    mlflow.log_param("features_used", ",".join(feature_cols))

    pipeline.fit(X_train, y_train)
    preds = pipeline.predict(X_test)

    mae = mean_absolute_error(y_test, preds)
    rmse = sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)

    mlflow.log_metric("mae", mae)
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("r2", r2)

    try:
        signature = infer_signature(X_train, pipeline.predict(X_train))

        mlflow.sklearn.log_model(
            sk_model=pipeline,
            artifact_path="model",
            signature=signature,
            registered_model_name=REGISTERED_MODEL_NAME
        )
        print(f"Model logged and registered as: {REGISTERED_MODEL_NAME}")
    except Exception as e:
        print("Model registry registration failed, logging artifact only.")
        print(f"Reason: {e}")

        mlflow.sklearn.log_model(
            sk_model=pipeline,
            artifact_path="model"
        )

    print("Run ID:", run.info.run_id)
    print({
        "mae": mae,
        "rmse": rmse,
        "r2": r2
    })